# High-Performance Renko Tick Calculation Engine

This notebook demonstrates the high-performance Renko Tick playback engine, featuring:
1. **Numeric Timestamp Normalization**: Automatically converts epoch strings or numbers of varying resolutions (s, ms, us, ns) to timezone-naive UTC datetimes.
2. **Dynamic JIT Array Resizing**: A JIT-compiled Numba loop that dynamically doubles its output arrays to prevent memory corruption when a price movement triggers more bricks than the tick count.

In [7]:
import math
import numpy as np
import pandas as pd
import polars as pl
import numba
from typing import Tuple, List, Dict, Any, Optional
from io import BytesIO

## 1. Datetime Normalization Helpers

In [8]:
def parse_csv_time_value(value: Any) -> pd.Timestamp | None:
    if value is None or pd.isna(value):
        return None
    raw_value = str(value).strip()
    if not raw_value:
        return None
    
    try:
        val = float(raw_value)
        if 1e9 <= val < 5e9:
            return pd.to_datetime(val, unit='s', utc=True)
        elif 1e12 <= val < 5e12:
            return pd.to_datetime(val, unit='ms', utc=True)
        elif 1e15 <= val < 5e15:
            return pd.to_datetime(val, unit='us', utc=True)
        elif 1e18 <= val < 5e18:
            return pd.to_datetime(val, unit='ns', utc=True)
    except ValueError:
        pass

    try:
        parsed = pd.Timestamp(raw_value)
        if parsed.tzinfo is None:
            parsed = parsed.tz_localize("UTC")
        else:
            parsed = parsed.tz_convert("UTC")
        return parsed
    except Exception:
        pass

    try:
        parsed_fallback = pd.to_datetime(raw_value, errors="coerce", utc=True)
        if not pd.isna(parsed_fallback):
            return pd.Timestamp(parsed_fallback)
    except Exception:
        pass
    return None

def normalize_polars_time_col(df: pl.DataFrame, time_col: str) -> pl.DataFrame:
    if df.is_empty():
        return df
    col_type = df[time_col].dtype
    if col_type.is_numeric():
        non_nulls = df[time_col].drop_nulls()
        if len(non_nulls) > 0:
            val = non_nulls[0]
            if 1e9 <= val < 5e9:
                df = df.with_columns((pl.col(time_col) * 1000).cast(pl.Datetime("ms")))
            elif 1e12 <= val < 5e12:
                df = df.with_columns(pl.col(time_col).cast(pl.Datetime("ms")))
            elif 1e15 <= val < 5e15:
                df = df.with_columns((pl.col(time_col) // 1000).cast(pl.Datetime("ms")))
            elif 1e18 <= val < 5e18:
                df = df.with_columns((pl.col(time_col) // 1000000).cast(pl.Datetime("ms")))
        return df
    
    if col_type == pl.String:
        non_nulls = df[time_col].drop_nulls()
        if len(non_nulls) > 0 and str(non_nulls[0]).strip().isdigit():
            df = df.with_columns(pl.col(time_col).cast(pl.Int64, strict=False))
            return normalize_polars_time_col(df, time_col)
        df = df.with_columns(pl.col(time_col).str.to_datetime(strict=False))
    return df

## 2. JIT Renko Calculation Engine

In [9]:
UP_FILL = "#22c55e"
UP_LINE = "#16a34a"
DOWN_FILL = "#fb7185"
DOWN_LINE = "#e11d48"

@numba.njit(nogil=True, cache=True)
def _build_renko_numba(
    prices: np.ndarray,
    brick_size: float,
    reversal_boxes: int,
    anchor_mode_int: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, int]:
    n = len(prices)
    init_size = max(1024, n)
    out_opens = np.zeros(init_size, dtype=np.float64)
    out_closes = np.zeros(init_size, dtype=np.float64)
    out_highs = np.zeros(init_size, dtype=np.float64)
    out_lows = np.zeros(init_size, dtype=np.float64)
    out_directions = np.zeros(init_size, dtype=np.int8)
    out_times_idx = np.zeros(init_size, dtype=np.int64)
    out_ticks = np.zeros(init_size, dtype=np.int32)
    
    if n == 0:
        return out_opens[:0], out_closes[:0], out_highs[:0], out_lows[:0], out_directions[:0], out_times_idx[:0], out_ticks[:0], 0
        
    last_close = 0.0
    direction = 0
    brick_idx = 0
    
    live_open = 0.0
    live_high = 0.0
    live_low = 0.0
    live_tick_count = 0
    
    has_first = False
    eps = brick_size / 1000000.0
    
    for i in range(n):
        price = prices[i]
        if not has_first:
            if anchor_mode_int == 2:
                last_close = price
            elif anchor_mode_int == 1:
                last_close = round(price / brick_size) * brick_size
            else:
                last_close = math.floor(price / brick_size) * brick_size
            live_open = last_close
            live_high = price
            live_low = price
            live_tick_count = 1
            has_first = True
            continue
            
        live_high = max(live_high, price)
        live_low = min(live_low, price)
        live_tick_count += 1
        
        while True:
            reversal = max(1, reversal_boxes)
            up_distance = brick_size if direction >= 0 else reversal * brick_size
            down_distance = brick_size if direction <= 0 else reversal * brick_size
            up_trigger = last_close + up_distance
            down_trigger = last_close - down_distance
            
            if price >= up_trigger - eps:
                brick_open = (last_close + (reversal - 1) * brick_size) if direction < 0 else last_close
                brick_close = brick_open + brick_size
                
                if brick_idx >= len(out_opens):
                    new_size = len(out_opens) * 2
                    new_opens = np.zeros(new_size, dtype=np.float64)
                    new_opens[:brick_idx] = out_opens[:brick_idx]
                    out_opens = new_opens
                    
                    new_closes = np.zeros(new_size, dtype=np.float64)
                    new_closes[:brick_idx] = out_closes[:brick_idx]
                    out_closes = new_closes
                    
                    new_highs = np.zeros(new_size, dtype=np.float64)
                    new_highs[:brick_idx] = out_highs[:brick_idx]
                    out_highs = new_highs
                    
                    new_lows = np.zeros(new_size, dtype=np.float64)
                    new_lows[:brick_idx] = out_lows[:brick_idx]
                    out_lows = new_lows
                    
                    new_directions = np.zeros(new_size, dtype=np.int8)
                    new_directions[:brick_idx] = out_directions[:brick_idx]
                    out_directions = new_directions
                    
                    new_times_idx = np.zeros(new_size, dtype=np.int64)
                    new_times_idx[:brick_idx] = out_times_idx[:brick_idx]
                    out_times_idx = new_times_idx
                    
                    new_ticks = np.zeros(new_size, dtype=np.int32)
                    new_ticks[:brick_idx] = out_ticks[:brick_idx]
                    out_ticks = new_ticks
                
                out_opens[brick_idx] = brick_open
                out_closes[brick_idx] = brick_close
                out_highs[brick_idx] = max(live_high, max(brick_open, brick_close))
                out_lows[brick_idx] = min(live_low, min(brick_open, brick_close))
                out_directions[brick_idx] = 1
                out_times_idx[brick_idx] = i
                out_ticks[brick_idx] = live_tick_count
                
                brick_idx += 1
                last_close = brick_close
                direction = 1
                
                live_open = last_close
                live_high = last_close
                live_low = last_close
                live_tick_count = 0
                continue
                
            if price <= down_trigger + eps:
                brick_open = (last_close - (reversal - 1) * brick_size) if direction > 0 else last_close
                brick_close = brick_open - brick_size
                
                if brick_idx >= len(out_opens):
                    new_size = len(out_opens) * 2
                    new_opens = np.zeros(new_size, dtype=np.float64)
                    new_opens[:brick_idx] = out_opens[:brick_idx]
                    out_opens = new_opens
                    
                    new_closes = np.zeros(new_size, dtype=np.float64)
                    new_closes[:brick_idx] = out_closes[:brick_idx]
                    out_closes = new_closes
                    
                    new_highs = np.zeros(new_size, dtype=np.float64)
                    new_highs[:brick_idx] = out_highs[:brick_idx]
                    out_highs = new_highs
                    
                    new_lows = np.zeros(new_size, dtype=np.float64)
                    new_lows[:brick_idx] = out_lows[:brick_idx]
                    out_lows = new_lows
                    
                    new_directions = np.zeros(new_size, dtype=np.int8)
                    new_directions[:brick_idx] = out_directions[:brick_idx]
                    out_directions = new_directions
                    
                    new_times_idx = np.zeros(new_size, dtype=np.int64)
                    new_times_idx[:brick_idx] = out_times_idx[:brick_idx]
                    out_times_idx = new_times_idx
                    
                    new_ticks = np.zeros(new_size, dtype=np.int32)
                    new_ticks[:brick_idx] = out_ticks[:brick_idx]
                    out_ticks = new_ticks
                
                out_opens[brick_idx] = brick_open
                out_closes[brick_idx] = brick_close
                out_highs[brick_idx] = max(live_high, max(brick_open, brick_close))
                out_lows[brick_idx] = min(live_low, min(brick_open, brick_close))
                out_directions[brick_idx] = -1
                out_times_idx[brick_idx] = i
                out_ticks[brick_idx] = live_tick_count
                
                brick_idx += 1
                last_close = brick_close
                direction = -1
                
                live_open = last_close
                live_high = last_close
                live_low = last_close
                live_tick_count = 0
                continue
                
            break
            
    return (
        out_opens[:brick_idx],
        out_closes[:brick_idx],
        out_highs[:brick_idx],
        out_lows[:brick_idx],
        out_directions[:brick_idx],
        out_times_idx[:brick_idx],
        out_ticks[:brick_idx],
        brick_idx
)

def build_renko_cpu(
    prices: np.ndarray,
    times: np.ndarray,
    brick_pips: float,
    reversal_boxes: int,
    pip_size: float,
    anchor_mode: str,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    brick_size = brick_pips * pip_size
    n = len(prices)
    if n == 0:
        empty = np.array([])
        return (empty, empty, empty, empty, empty, empty, empty, empty, empty, empty, empty, empty)
        
    anchor_mode_int = 0
    if anchor_mode == "round":
        anchor_mode_int = 1
    elif anchor_mode == "first":
        anchor_mode_int = 2
        
    (opens, closes, highs, lows, directions, times_idx, ticks, brick_count) = _build_renko_numba(
        prices, brick_size, reversal_boxes, anchor_mode_int
    )
    
    if brick_count == 0:
        empty = np.array([])
        return (empty, empty, empty, empty, empty, empty, empty, empty, empty, empty, empty, empty)
        
    tops = np.maximum(opens, closes)
    bottoms = np.minimum(opens, closes)
    indices = np.arange(brick_count, dtype=np.int32)
    
    out_colors = np.where(directions == 1, UP_FILL, DOWN_FILL)
    out_borders = np.where(directions == 1, UP_LINE, DOWN_LINE)
    out_directions_str = np.where(directions == 1, "up", "down")
    out_times = times[times_idx]
    
    return (
        indices, opens, closes, tops, bottoms, highs, lows,
        out_colors, out_borders, out_times, ticks, out_directions_str
    )

## 3. Demo Run on a Mock Dataset

In [ ]:
# Generate 10,000 synthetic ticks with random walk
np.random.seed(42)
steps = np.random.normal(0, 0.0002, 10000)
prices = 1.1700 + np.cumsum(steps)

# Generate string epoch timestamps in milliseconds
start_time = 1767304920000
times_epoch = np.array([str(start_time + i * 1000) for i in range(10000)], dtype=object)

print(f"Mock data generated: {len(prices)} ticks")
print("First 5 prices:", prices[:5])
print("First 5 timestamps:", times_epoch[:5])

In [ ]:
# Normalize epoch timestamps to timezone-naive datetimes via helper
df_pl = pl.DataFrame({"timestamp": times_epoch})
df_pl = normalize_polars_time_col(df_pl, "timestamp")

# Format back to numpy string format to match build_renko_cpu expectations
times_normalized = df_pl["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S.%f").to_numpy()
times_normalized = np.array([t[:-3] if len(t) > 23 else t for t in times_normalized], dtype=object)

print("First 5 normalized datetimes:", times_normalized[:5])

In [ ]:
# Run the JIT Renko calculation (1.0 Pip sizes)
( 
    indices, opens, closes, tops, bottoms, highs, lows,
    colors, borders, times_out, ticks, directions
) = build_renko_cpu(prices, times_normalized, 1.0, 2, 0.0001, "floor")

print(f"Calculation completed successfully!")
print(f"Built {len(opens)} Renko bricks.")
if len(opens) > 0:
    print("First brick:", {
        "time": indices[0] + 1,
        "confirm_time": times_out[0],
        "open": opens[0],
        "close": closes[0],
        "direction": directions[0],
        "tick_count": ticks[0]
    })